# Trendline Detector — Swing-Anchored Trendlines

Connects swing highs to swing highs and swing lows to swing lows,
validating that no intermediate bar body breaches the line by more than 0.5 ATR.

**Pairings evaluated:**
- HIGH (active/expired) → ACTIVE HIGH
- LOW (active/expired) → ACTIVE LOW

**Additional downtrendline view:**
- ACTIVE HIGH → ACTIVE HIGH (negative slope only)

**Interception rule:** body-only (max(open,close) for high-lines, min(open,close) for low-lines)

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass, field
from typing import Literal

## 1 — Fetch data

In [11]:
TICKER = "QQQ"
PERIOD = "2y"
INTERVAL = "1d"

raw = yf.Ticker(TICKER).history(period=PERIOD, interval=INTERVAL)
if raw.empty:
    raise ValueError("No data returned from yfinance.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
raw.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)

df_all = raw.rename(columns={
    "Open": "open", "High": "high", "Low": "low",
    "Close": "close", "Volume": "volume",
})

print(f"Fetched {len(df_all)} bars for {TICKER}")
df_all.tail()

Fetched 502 bars for QQQ


,date,open,high,low,close,volume
497,2026-02-19 00:00:00-05:00,602.809998,605.820007,600.750000,603.469971,60960800
498,2026-02-20 00:00:00-05:00,600.119995,610.349976,599.229980,608.809998,74127300
499,2026-02-23 00:00:00-05:00,606.609985,608.010010,599.049988,601.409973,63859100
500,2026-02-24 00:00:00-05:00,602.400024,608.989990,599.729980,607.869995,55023700
501,2026-02-25 00:00:00-05:00,611.090027,616.830017,611.000000,616.679993,54001210


## 2 — ATR & Indicators

In [12]:
def true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr1 = df["high"] - df["low"]
    tr2 = (df["high"] - prev_close).abs()
    tr3 = (df["low"] - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def atr_series(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Simple rolling-mean ATR (matches the TS backend)."""
    tr = true_range(df)
    return tr.rolling(window=period, min_periods=period).mean()


def average_bar_size(df: pd.DataFrame, period: int = 20) -> pd.Series:
    return (df["high"] - df["low"]).rolling(window=period, min_periods=1).mean()

## 3 — Swing Point Detection

In [13]:
@dataclass
class SwingPoint:
    index: int
    price: float
    type: Literal["HIGH", "LOW"]
    atr: float = 0.0
    prominence: float = 0.0


def detect_fractal_pivots(
    df: pd.DataFrame, lookahead: int = 10,
) -> list[SwingPoint]:
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = df["atr14"].values
    abs_vals = df["abs20"].values
    n = len(df)
    points: list[SwingPoint] = []

    for t in range(n - lookahead):
        abs_tol = abs_vals[t] if np.isfinite(abs_vals[t]) else 0
        a = atr_vals[t] if np.isfinite(atr_vals[t]) else abs_tol

        if all(highs[i] < highs[t] - abs_tol for i in range(t + 1, t + lookahead + 1)):
            points.append(SwingPoint(index=t, price=highs[t], type="HIGH", atr=a))

        if all(lows[i] > lows[t] + abs_tol for i in range(t + 1, t + lookahead + 1)):
            points.append(SwingPoint(index=t, price=lows[t], type="LOW", atr=a))

    return points


def detect_significant_swings(
    df: pd.DataFrame,
    left: int = 3, right: int = 3,
    atr_period: int = 14,
    prom_atr: float = 1.5,
    depart_atr: float = 2.5,
    depart_lookahead: int = 10,
    min_swing_sep: int = 7,
) -> list[SwingPoint]:
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = atr_series(df, atr_period).values
    n = len(df)
    candidates: list[SwingPoint] = []

    for t in range(left, n - right):
        a = atr_vals[t]
        if not np.isfinite(a) or a <= 0:
            continue
        window = slice(t - left, t + right + 1)

        if highs[t] == highs[window].max() and all(
            highs[i] < highs[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_min_low = lows[window].min()
            prominence = highs[t] - local_min_low
            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                min_low_after = lows[t + 1 : dep_end].min() if t + 1 < dep_end else np.inf
                if min_low_after <= highs[t] - depart_atr * a:
                    candidates.append(SwingPoint(t, highs[t], "HIGH", a, prominence))

        if lows[t] == lows[window].min() and all(
            lows[i] > lows[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_max_high = highs[window].max()
            prominence = local_max_high - lows[t]
            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                max_high_after = highs[t + 1 : dep_end].max() if t + 1 < dep_end else -np.inf
                if max_high_after >= lows[t] + depart_atr * a:
                    candidates.append(SwingPoint(t, lows[t], "LOW", a, prominence))

    candidates.sort(key=lambda p: p.index)
    result: list[SwingPoint] = []
    for p in candidates:
        if not result:
            result.append(p)
            continue
        last = result[-1]
        if p.type == last.type and p.index - last.index <= min_swing_sep:
            keep_new = p.price > last.price if p.type == "HIGH" else p.price < last.price
            if keep_new:
                result[-1] = p
        else:
            result.append(p)
    return result

## 4 — Active / Expired Swing Classification

In [14]:
def split_active_expired_swings(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    latest_atr: float,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """
    Path-based expiration:
      - HIGH expires if any future HIGH breaches swing_high + atr_buffer*ATR_ref
      - LOW  expires if any future LOW  breaches swing_low  - atr_buffer*ATR_ref
    Returns: (active, expired_high, expired_low)
    """
    active: list[SwingPoint] = []
    expired_high: list[SwingPoint] = []
    expired_low: list[SwingPoint] = []

    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values
    n = len(df_ctx)

    for s in swings:
        if s.index >= n - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = latest_atr if np.isfinite(latest_atr) and latest_atr > 0 else 0.0

        thr = atr_buffer * atr_ref

        if s.type == "HIGH":
            breached = np.any(highs[s.index + 1 :] > s.price + thr)
            (expired_high if breached else active).append(s)
        else:
            breached = np.any(lows[s.index + 1 :] < s.price - thr)
            (expired_low if breached else active).append(s)

    return active, expired_high, expired_low

## 5 — Configuration & Data Preparation

In [15]:
# ── Date range (None = full range) ──
START_DATE = None
END_DATE = None

# ── Swing detection params ──
FRACTAL_LOOKAHEAD = 10
SIG_LEFT = 3
SIG_RIGHT = 3
SIG_PROM_ATR = 1.5
SIG_DEPART_ATR = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP = 7

# ── Trendline params ──
TL_ATR_TOLERANCE = 0.5   # max body breach beyond the line, in ATR units


def apply_date_range(df_src, start_date=None, end_date=None):
    out = df_src.copy()
    if start_date is not None:
        out = out[out["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        out = out[out["date"] <= pd.to_datetime(end_date)]
    return out.reset_index(drop=True)


def add_indicators(df_in):
    out = df_in.copy()
    out["atr14"] = atr_series(out, 14)
    out["abs20"] = average_bar_size(out, 20)
    out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    out["sma50"] = out["close"].rolling(50).mean()
    return out


# Prepare data
df = apply_date_range(df_all, START_DATE, END_DATE)
if len(df) < 60:
    raise ValueError("Date range too small — need at least ~60 bars.")
df = add_indicators(df)

latest_atr = float(df["atr14"].iloc[-1]) if np.isfinite(df["atr14"].iloc[-1]) else 0.0

# Detect swings
fractal_pivots = detect_fractal_pivots(df, lookahead=FRACTAL_LOOKAHEAD)
sig_swings = detect_significant_swings(
    df, left=SIG_LEFT, right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR, depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK, min_swing_sep=SIG_MIN_SEP,
)

# Merge & deduplicate swings from both detectors
all_swings_raw = fractal_pivots + sig_swings
seen = set()
all_swings: list[SwingPoint] = []
for s in sorted(all_swings_raw, key=lambda p: (p.index, p.type)):
    key = (s.index, s.type)
    if key not in seen:
        seen.add(key)
        all_swings.append(s)

# Split into active / expired
active, expired_high, expired_low = split_active_expired_swings(
    all_swings, df, latest_atr, atr_buffer=1.0,
)

active_highs = [s for s in active if s.type == "HIGH"]
active_lows = [s for s in active if s.type == "LOW"]
all_highs = active_highs + expired_high
all_lows = active_lows + expired_low

range_text = f"[{df['date'].iloc[0]} → {df['date'].iloc[-1]}]"
print(f"Bars: {len(df)}  {range_text}")
print(f"Latest ATR14: {latest_atr:.2f}")
print(f"All highs: {len(all_highs)} (active={len(active_highs)}, expired={len(expired_high)})")
print(f"All lows:  {len(all_lows)} (active={len(active_lows)}, expired={len(expired_low)})")

Bars: 502  [2024-02-26 00:00:00-05:00 → 2026-02-25 00:00:00-05:00]
Latest ATR14: 10.20
All highs: 31 (active=6, expired=25)
All lows:  40 (active=13, expired=27)


## 6 — Trendline Detection Core

In [16]:
@dataclass
class Trendline:
    anchor: SwingPoint
    endpoint: SwingPoint
    line_type: Literal["HIGH", "LOW"]
    slope: float
    span: int
    max_breach_atr: float

    def price_at(self, idx: int) -> float:
        return self.anchor.price + self.slope * (idx - self.anchor.index)


def _validate_trendline(
    anchor: SwingPoint,
    endpoint: SwingPoint,
    line_type: Literal["HIGH", "LOW"],
    opens: np.ndarray,
    closes: np.ndarray,
    atr_vals: np.ndarray,
    tol_atr: float,
    extend_to_latest: bool = True,
) -> Trendline | None:
    """
    Validate a line with body-only interception rules:
    - HIGH line invalid if max(open,close) > line + tol_atr*ATR
    - LOW line invalid if min(open,close) < line - tol_atr*ATR

    If extend_to_latest=True, extend the line to the latest bar and require
    no post-endpoint breaches as well.
    """
    i0, i1 = anchor.index, endpoint.index
    if i1 <= i0:
        return None

    n = len(opens)
    span = i1 - i0
    slope = (endpoint.price - anchor.price) / span

    max_breach = 0.0

    # 1) Check interior bars between anchors
    for k in range(i0 + 1, i1):
        line_price = anchor.price + slope * (k - i0)
        atr_k = atr_vals[k]
        if not np.isfinite(atr_k) or atr_k <= 0:
            continue

        if line_type == "HIGH":
            body_top = max(opens[k], closes[k])
            breach = body_top - line_price
        else:
            body_bottom = min(opens[k], closes[k])
            breach = line_price - body_bottom

        if breach > 0:
            breach_atr = breach / atr_k
            if breach_atr > tol_atr:
                return None
            max_breach = max(max_breach, breach_atr)

    # 2) Extend line to the right and ensure no future breaches
    if extend_to_latest and i1 + 1 < n:
        for k in range(i1 + 1, n):
            line_price = anchor.price + slope * (k - i0)
            atr_k = atr_vals[k]
            if not np.isfinite(atr_k) or atr_k <= 0:
                continue

            if line_type == "HIGH":
                body_top = max(opens[k], closes[k])
                breach = body_top - line_price
            else:
                body_bottom = min(opens[k], closes[k])
                breach = line_price - body_bottom

            if breach > 0:
                breach_atr = breach / atr_k
                if breach_atr > tol_atr:
                    return None
                max_breach = max(max_breach, breach_atr)

    return Trendline(
        anchor=anchor,
        endpoint=endpoint,
        line_type=line_type,
        slope=slope,
        span=span,
        max_breach_atr=round(max_breach, 4),
    )


def detect_downtrendlines_active_highs(
    df: pd.DataFrame,
    active_highs: list[SwingPoint],
    tol_atr: float = 0.5,
) -> tuple[list[Trendline], dict]:
    """
    Down trendlines:
    - ACTIVE HIGH -> ACTIVE HIGH
    - Negative slope only
    - Must remain clean when extended to latest bar
    """
    opens = df["open"].values
    closes = df["close"].values
    atr_vals = df["atr14"].values

    stats = {"down_candidates": 0, "down_accepted": 0, "hidden_upward_high_high": 0}
    down_lines: list[Trendline] = []

    ordered_highs = sorted(active_highs, key=lambda s: s.index)
    for ep in ordered_highs:
        for anchor in ordered_highs:
            if anchor.index >= ep.index:
                continue
            stats["down_candidates"] += 1

            raw_slope = (ep.price - anchor.price) / (ep.index - anchor.index)
            if raw_slope >= 0:
                stats["hidden_upward_high_high"] += 1
                continue

            tl = _validate_trendline(
                anchor,
                ep,
                "HIGH",
                opens,
                closes,
                atr_vals,
                tol_atr,
                extend_to_latest=True,
            )
            if tl is None:
                continue
            down_lines.append(tl)
            stats["down_accepted"] += 1

    down_lines.sort(key=lambda t: (-t.span, t.max_breach_atr))
    return down_lines, stats


def detect_uptrendlines_active_lows(
    df: pd.DataFrame,
    active_lows: list[SwingPoint],
    tol_atr: float = 0.5,
) -> tuple[list[Trendline], dict]:
    """
    Up trendlines:
    - ACTIVE LOW -> ACTIVE LOW
    - Positive slope only
    - Must remain clean when extended to latest bar
    """
    opens = df["open"].values
    closes = df["close"].values
    atr_vals = df["atr14"].values

    stats = {"up_candidates": 0, "up_accepted": 0, "hidden_downward_low_low": 0}
    up_lines: list[Trendline] = []

    ordered_lows = sorted(active_lows, key=lambda s: s.index)
    for ep in ordered_lows:
        for anchor in ordered_lows:
            if anchor.index >= ep.index:
                continue
            stats["up_candidates"] += 1

            raw_slope = (ep.price - anchor.price) / (ep.index - anchor.index)
            if raw_slope <= 0:
                stats["hidden_downward_low_low"] += 1
                continue

            tl = _validate_trendline(
                anchor,
                ep,
                "LOW",
                opens,
                closes,
                atr_vals,
                tol_atr,
                extend_to_latest=True,
            )
            if tl is None:
                continue
            up_lines.append(tl)
            stats["up_accepted"] += 1

    up_lines.sort(key=lambda t: (-t.span, t.max_breach_atr))
    return up_lines, stats

## 7 — Run Detection & Diagnostics

In [17]:
down_high_lines, down_stats = detect_downtrendlines_active_highs(
    df, active_highs, tol_atr=TL_ATR_TOLERANCE,
)
up_low_lines, up_stats = detect_uptrendlines_active_lows(
    df, active_lows, tol_atr=TL_ATR_TOLERANCE,
)

# Compatibility aliases for downstream cells
high_lines = down_high_lines
low_lines = up_low_lines

print("=== Trendline Detection Results (Extended to Latest Bar) ===")
print(f"DOWN lines (active HIGH→active HIGH): {down_stats['down_candidates']} candidates → {down_stats['down_accepted']} accepted")
print(f"  hidden upward HIGH→HIGH: {down_stats['hidden_upward_high_high']}")
print(f"UP   lines (active LOW→active LOW):   {up_stats['up_candidates']} candidates → {up_stats['up_accepted']} accepted")
print(f"  hidden downward LOW→LOW:  {up_stats['hidden_downward_low_low']}")

MAX_SHOW = 5
if down_high_lines:
    print(f"\nTop {min(MAX_SHOW, len(down_high_lines))} DOWN trendlines (active HIGH→active HIGH):")
    for tl in down_high_lines[:MAX_SHOW]:
        d0 = df["date"].iloc[tl.anchor.index]
        d1 = df["date"].iloc[tl.endpoint.index]
        print(f"  [{tl.anchor.index}→{tl.endpoint.index}] {d0} → {d1}  "
              f"span={tl.span}  slope={tl.slope:.4f}  max_breach={tl.max_breach_atr:.3f} ATR  "
              f"anchor=${tl.anchor.price:.2f}  end=${tl.endpoint.price:.2f}")

if up_low_lines:
    print(f"\nTop {min(MAX_SHOW, len(up_low_lines))} UP trendlines (active LOW→active LOW):")
    for tl in up_low_lines[:MAX_SHOW]:
        d0 = df["date"].iloc[tl.anchor.index]
        d1 = df["date"].iloc[tl.endpoint.index]
        print(f"  [{tl.anchor.index}→{tl.endpoint.index}] {d0} → {d1}  "
              f"span={tl.span}  slope={tl.slope:.4f}  max_breach={tl.max_breach_atr:.3f} ATR  "
              f"anchor=${tl.anchor.price:.2f}  end=${tl.endpoint.price:.2f}")

=== Trendline Detection Results ===
HIGH lines (all anchors): 163 candidates → 43 accepted
LOW  lines (all anchors): 373 candidates → 86 accepted
DOWN lines (active HIGH→active HIGH): 15 candidates → 6 accepted

Top 5 DOWN trendlines (active HIGH→active HIGH):
  [421→474] 2025-10-29 00:00:00-04:00 → 2026-01-15 00:00:00-05:00  span=53  slope=-0.1168  max_breach=0.000 ATR  anchor=$636.19  end=$630.00
  [424→474] 2025-11-03 00:00:00-05:00 → 2026-01-15 00:00:00-05:00  span=50  slope=-0.1000  max_breach=0.000 ATR  anchor=$635.00  end=$630.00
  [421→450] 2025-10-29 00:00:00-04:00 → 2025-12-10 00:00:00-05:00  span=29  slope=-0.2686  max_breach=0.000 ATR  anchor=$636.19  end=$628.40
  [424→450] 2025-11-03 00:00:00-05:00 → 2025-12-10 00:00:00-05:00  span=26  slope=-0.2539  max_breach=0.000 ATR  anchor=$635.00  end=$628.40
  [482→486] 2026-01-28 00:00:00-05:00 → 2026-02-03 00:00:00-05:00  span=4  slope=-1.6550  max_breach=0.000 ATR  anchor=$636.60  end=$629.98

Top 5 HIGH trendlines (by span):
 

## 8 — Visualization

In [18]:
# Limit lines drawn to avoid clutter
MAX_DOWN_LINES = 5
MAX_UP_LINES = 5

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.76, 0.24], vertical_spacing=0.03)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df["date"], open=df["open"], high=df["high"],
    low=df["low"], close=df["close"], name="Price",
), row=1, col=1)

# MAs
fig.add_trace(go.Scatter(x=df["date"], y=df["ema20"], name="EMA 20",
    line=dict(color="#f59e0b", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["sma50"], name="SMA 50",
    line=dict(color="#3b82f6", width=1.5)), row=1, col=1)

# Active swing highs
if active_highs:
    fig.add_trace(go.Scatter(
        x=[df["date"].iloc[s.index] for s in active_highs],
        y=[s.price for s in active_highs],
        mode="markers", name="Active Swing High",
        marker=dict(symbol="star", size=10, color="#dc2626",
                    line=dict(width=1, color="white")),
    ), row=1, col=1)

# Active swing lows
if active_lows:
    fig.add_trace(go.Scatter(
        x=[df["date"].iloc[s.index] for s in active_lows],
        y=[s.price for s in active_lows],
        mode="markers", name="Active Swing Low",
        marker=dict(symbol="star", size=10, color="#16a34a",
                    line=dict(width=1, color="white")),
    ), row=1, col=1)

# Expired swing highs
if expired_high:
    fig.add_trace(go.Scatter(
        x=[df["date"].iloc[s.index] for s in expired_high],
        y=[s.price for s in expired_high],
        mode="markers", name="Expired Swing High",
        marker=dict(symbol="x", size=8, color="rgba(239,68,68,0.60)"),
    ), row=1, col=1)

# Expired swing lows
if expired_low:
    fig.add_trace(go.Scatter(
        x=[df["date"].iloc[s.index] for s in expired_low],
        y=[s.price for s in expired_low],
        mode="markers", name="Expired Swing Low",
        marker=dict(symbol="x", size=8, color="rgba(34,197,94,0.60)"),
    ), row=1, col=1)

# DOWN trendlines (active HIGH→active HIGH, negative slope only)
down_colors = ["#a855f7", "#8b5cf6", "#7c3aed", "#c084fc", "#d8b4fe"]
for i, tl in enumerate(down_high_lines[:MAX_DOWN_LINES]):
    color = down_colors[i % len(down_colors)]
    x0 = df["date"].iloc[tl.anchor.index]
    x1 = df["date"].iloc[tl.endpoint.index]
    y0 = tl.anchor.price
    y1 = tl.endpoint.price
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1], mode="lines+markers",
        name=f"Down H-H {tl.anchor.index}→{tl.endpoint.index} (span={tl.span})",
        line=dict(color=color, width=2.2),
        marker=dict(size=6, color=color),
    ), row=1, col=1)

# UP trendlines (active LOW→active LOW, positive slope only)
up_colors = ["#22c55e", "#14b8a6", "#2dd4bf", "#5eead4", "#99f6e4"]
for i, tl in enumerate(up_low_lines[:MAX_UP_LINES]):
    color = up_colors[i % len(up_colors)]
    x0 = df["date"].iloc[tl.anchor.index]
    x1 = df["date"].iloc[tl.endpoint.index]
    y0 = tl.anchor.price
    y1 = tl.endpoint.price
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1], mode="lines+markers",
        name=f"Up L-L {tl.anchor.index}→{tl.endpoint.index} (span={tl.span})",
        line=dict(color=color, width=2),
        marker=dict(size=6, color=color),
    ), row=1, col=1)

# Volume
vol_colors = ["#22c55e" if df["close"].iloc[i] >= df["open"].iloc[i] else "#ef4444" for i in range(len(df))]
fig.add_trace(go.Bar(
    x=df["date"], y=df["volume"], name="Volume",
    marker_color=vol_colors, opacity=0.45,
), row=2, col=1)

fig.update_layout(
    title=f"{TICKER} — Trendline Detector  {range_text}",
    template="plotly_dark",
    height=860,
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show()

## 9 — Validation Spot-Check

Pick a few accepted and rejected trendlines and verify the body-only interception rule.

In [19]:
def spot_check_line(tl: Trendline, df: pd.DataFrame, label: str = ""):
    """Print per-bar breach info for a trendline."""
    opens = df["open"].values
    closes = df["close"].values
    atr_vals = df["atr14"].values
    i0, i1 = tl.anchor.index, tl.endpoint.index
    print(f"\n--- {label} [{i0}→{i1}] type={tl.line_type} span={tl.span} ---")
    print(f"  anchor=${tl.anchor.price:.2f}  end=${tl.endpoint.price:.2f}  slope={tl.slope:.4f}")

    worst_breach = 0.0
    for k in range(i0 + 1, i1):
        line_p = tl.price_at(k)
        atr_k = atr_vals[k]
        if not np.isfinite(atr_k) or atr_k <= 0:
            continue
        if tl.line_type == "HIGH":
            body = max(opens[k], closes[k])
            breach = body - line_p
        else:
            body = min(opens[k], closes[k])
            breach = line_p - body
        breach_atr = breach / atr_k if breach > 0 else 0.0
        worst_breach = max(worst_breach, breach_atr)
        if breach_atr > 0.1:
            print(f"    bar {k}: body={body:.2f}  line={line_p:.2f}  breach={breach:.2f}  ({breach_atr:.3f} ATR)")

    status = "PASS" if worst_breach <= TL_ATR_TOLERANCE else "FAIL"
    print(f"  worst_breach={worst_breach:.4f} ATR  → {status}")


CHECK_COUNT = 3

print("=== Spot-check accepted HIGH trendlines ===")
for tl in high_lines[:CHECK_COUNT]:
    spot_check_line(tl, df, "ACCEPTED HIGH")

print("\n=== Spot-check accepted LOW trendlines ===")
for tl in low_lines[:CHECK_COUNT]:
    spot_check_line(tl, df, "ACCEPTED LOW")

# Show a few rejected candidates for comparison
print("\n=== Spot-check rejected candidates ===")
rejected_count = 0
opens_arr = df["open"].values
closes_arr = df["close"].values
atr_arr = df["atr14"].values

for ep in active_highs:
    if rejected_count >= CHECK_COUNT:
        break
    for anchor in all_highs:
        if anchor.index >= ep.index:
            continue
        tl = _validate_trendline(anchor, ep, "HIGH", opens_arr, closes_arr, atr_arr, TL_ATR_TOLERANCE)
        if tl is None:
            rejected_tl = Trendline(
                anchor=anchor, endpoint=ep, line_type="HIGH",
                slope=(ep.price - anchor.price) / (ep.index - anchor.index),
                span=ep.index - anchor.index, max_breach_atr=999.0,
            )
            spot_check_line(rejected_tl, df, "REJECTED HIGH")
            rejected_count += 1
            break

for ep in active_lows:
    if rejected_count >= CHECK_COUNT * 2:
        break
    for anchor in all_lows:
        if anchor.index >= ep.index:
            continue
        tl = _validate_trendline(anchor, ep, "LOW", opens_arr, closes_arr, atr_arr, TL_ATR_TOLERANCE)
        if tl is None:
            rejected_tl = Trendline(
                anchor=anchor, endpoint=ep, line_type="LOW",
                slope=(ep.price - anchor.price) / (ep.index - anchor.index),
                span=ep.index - anchor.index, max_breach_atr=999.0,
            )
            spot_check_line(rejected_tl, df, "REJECTED LOW")
            rejected_count += 1
            break

print(f"\nDone. Total accepted: {len(high_lines)} HIGH + {len(low_lines)} LOW = {len(high_lines)+len(low_lines)} trendlines.")

=== Spot-check accepted HIGH trendlines ===

--- ACCEPTED HIGH [93→424] type=HIGH span=331 ---
  anchor=$499.47  end=$635.00  slope=0.4095
    bar 421: body=634.95  line=633.77  breach=1.18  (0.113 ATR)
  worst_breach=0.1129 ATR  → PASS

--- ACCEPTED HIGH [93→421] type=HIGH span=328 ---
  anchor=$499.47  end=$636.19  slope=0.4168
  worst_breach=0.0000 ATR  → PASS

--- ACCEPTED HIGH [97→424] type=HIGH span=327 ---
  anchor=$494.43  end=$635.00  slope=0.4299
    bar 421: body=634.95  line=633.71  breach=1.24  (0.119 ATR)
  worst_breach=0.1188 ATR  → PASS

=== Spot-check accepted LOW trendlines ===

--- ACCEPTED LOW [38→279] type=LOW span=241 ---
  anchor=$409.10  end=$400.96  slope=-0.0338
  worst_breach=0.0000 ATR  → PASS

--- ACCEPTED LOW [47→279] type=LOW span=232 ---
  anchor=$416.59  end=$400.96  slope=-0.0674
  worst_breach=0.0000 ATR  → PASS

--- ACCEPTED LOW [279→455] type=LOW span=176 ---
  anchor=$400.96  end=$599.51  slope=1.1281
  worst_breach=0.0000 ATR  → PASS

=== Spot-che